# SUFE VOS SAM2 Optimization Colab

This notebook runs the next optimization stage after the SAM2.1 Hiera Large baseline: conservative pseudo-anchor repropagation, weighted fusion, postprocess, diagnostics, and submission validation.

Recommended runtime: Colab Pro/Pro+ with A100 40GB is preferred. L4 usually works with original resolution and one anchor per video, but may be slower. T4 is only a fallback; if it OOMs, set `RESIZE_LONG_SIDE = 1536`, expect possible score loss, or run fewer videos for smoke tests first.

Use `MAX_VIDEOS = 1` only for smoke tests. Full leaderboard submission should use `MAX_VIDEOS = 0`.

In [ ]:
from __future__ import annotations

import datetime as dt
import json
import os
import re
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

IS_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = Path(os.environ.get("SUFE_DRIVE_ROOT", "/content/drive/MyDrive"))
def clean_repo_url(value: str) -> str:
    text = str(value or "").strip()
    markdown = re.fullmatch(r"\[([^\]]+)\]\(([^)]+)\)", text)
    if markdown:
        text = markdown.group(2).strip()
    if text.startswith("<") and text.endswith(">"):
        text = text[1:-1].strip()
    return text

def run_git(cmd: list[str], label: str) -> subprocess.CompletedProcess:
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with exit code {result.returncode}: {cmd}")
    return result

REPO_URL = clean_repo_url(os.environ.get("SUFE_REPO_URL", "https://github.com/yyj11266/SUFE_DL_FinalProject_Vision.git"))
DEFAULT_PROJECT_ROOT = Path("/content/sufe_vos_leaderboard") if IS_COLAB else Path.cwd()
PROJECT_ROOT = Path(os.environ.get("SUFE_PROJECT_ROOT", str(DEFAULT_PROJECT_ROOT))).expanduser().resolve()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if (PROJECT_ROOT / ".git").exists():
        run_git(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], "git pull")
    elif not PROJECT_ROOT.exists() and REPO_URL:
        run_git(["git", "clone", REPO_URL, str(PROJECT_ROOT)], "git clone")
    elif PROJECT_ROOT.exists() and not any(PROJECT_ROOT.iterdir()) and REPO_URL:
        PROJECT_ROOT.rmdir()
        run_git(["git", "clone", REPO_URL, str(PROJECT_ROOT)], "git clone")

if not (PROJECT_ROOT / "src" / "data" / "inspect_sufe.py").exists():
    raise RuntimeError(f"PROJECT_ROOT is wrong: {PROJECT_ROOT}. Set SUFE_PROJECT_ROOT before rerunning.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DATA_URL = "https://drive.google.com/file/d/12PLrZwDvpeO3n-IQbAMgA9FOM0xdOqRr/view?usp=sharing"
DEFAULT_DATA_ZIP = DRIVE_ROOT / "sufe_vos_inputs" / "video_dataset.zip" if IS_COLAB else PROJECT_ROOT / "data" / "video_dataset.zip"
DATA_ZIP = Path(os.environ.get("SUFE_DATA_ZIP", str(DEFAULT_DATA_ZIP))).expanduser().resolve()
DEFAULT_DATA_ROOT = Path("/content/sufe_data/video_dataset") if IS_COLAB else PROJECT_ROOT / "data" / "video_dataset"
DATA_ROOT = Path(os.environ.get("SUFE_DATA_ROOT", str(DEFAULT_DATA_ROOT))).expanduser().resolve()
SAMPLE_SUBMISSION_ZIP = Path(os.environ.get("SUFE_SAMPLE_SUBMISSION_ZIP", str(DRIVE_ROOT / "sufe_vos_inputs" / "sample_submission.zip"))).expanduser().resolve()
DEFAULT_OUTPUTS_ROOT = Path("/content/sufe_runs") if IS_COLAB else PROJECT_ROOT / "outputs"
OUTPUTS_ROOT = Path(os.environ.get("SUFE_OUTPUTS_ROOT", str(DEFAULT_OUTPUTS_ROOT))).expanduser().resolve()
PUBLISH_ROOT = Path(os.environ.get("SUFE_PUBLISH_ROOT", str(DRIVE_ROOT / "sufe_vos_review" / "runs"))).expanduser().resolve()
ARCHIVE_ROOT = Path(os.environ.get("SUFE_ARCHIVE_ROOT", str(DRIVE_ROOT / "sufe_vos_archives"))).expanduser().resolve()
DEFAULT_SAM2_REPO_DIR = Path("/content/sam2") if IS_COLAB else Path("/tmp/sufe_sam2")
SAM2_REPO_DIR = Path(os.environ.get("SUFE_SAM2_REPO_DIR", str(DEFAULT_SAM2_REPO_DIR))).expanduser().resolve()

BASELINE_EXP_ID_ENV = os.environ.get("SUFE_BASELINE_EXP_ID")
BASELINE_EXP_ID = BASELINE_EXP_ID_ENV or "sam21_hiera_large_full"
ENHANCED_EXP_ID = os.environ.get("SUFE_ENHANCED_EXP_ID", f"sam2_anchor_fusion_1x_{dt.datetime.now():%Y%m%d_%H%M%S}")

MAX_VIDEOS = int(os.environ.get("SUFE_MAX_VIDEOS", "0"))
RESIZE_LONG_SIDE = int(os.environ.get("SUFE_RESIZE_LONG_SIDE", "0"))
MAX_ANCHOR_RUNS = int(os.environ.get("SUFE_MAX_ANCHOR_RUNS", "1"))
ANCHOR_FRACTIONS = os.environ.get("SUFE_ANCHOR_FRACTIONS", "0.50")
REQUIRE_ANCHOR_RERUN = os.environ.get("SUFE_REQUIRE_ANCHOR_RERUN", "1").strip().lower() not in {"0", "false", "no"}

DATA_ZIP.parent.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
PUBLISH_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)

def experiment_complete(exp_dir: Path, require_full: bool = False) -> bool:
    if not ((exp_dir / "submission.zip").exists() and (exp_dir / "masks").exists()):
        return False
    sanity_path = exp_dir / "sanity_check.json"
    if not sanity_path.exists():
        return True
    try:
        sanity = json.loads(sanity_path.read_text(encoding="utf-8"))
    except Exception:
        return True
    if sanity.get("passed") is False:
        return False
    if require_full and int(sanity.get("num_actual_masks", 0)) < 1000:
        return False
    return True

def is_baseline_candidate(exp: Path) -> bool:
    name = exp.name.lower()
    if name.startswith(("sam2_anchor_fusion", "sam2_postprocess", "smoke_")):
        return False
    status_path = exp / "logs" / "per_video_status.json"
    if status_path.exists():
        try:
            summary = json.loads(status_path.read_text(encoding="utf-8")).get("summary", {})
            if "anchor_run_attempts" in summary or "anchor_run_successes" in summary:
                return False
        except Exception:
            pass
    return True

def discover_existing_baseline() -> Path | None:
    candidates = []
    for exp in OUTPUTS_ROOT.iterdir() if OUTPUTS_ROOT.exists() else []:
        if not exp.is_dir() or not is_baseline_candidate(exp):
            continue
        if not experiment_complete(exp, require_full=(MAX_VIDEOS == 0)):
            continue
        sanity_path = exp / "sanity_check.json"
        mask_count = 0
        passed = False
        if sanity_path.exists():
            try:
                sanity = json.loads(sanity_path.read_text(encoding="utf-8"))
                mask_count = int(sanity.get("num_actual_masks", 0))
                passed = bool(sanity.get("passed"))
            except Exception:
                pass
        candidates.append((passed, mask_count, exp.stat().st_mtime, exp))
    if not candidates:
        return None
    return sorted(candidates, reverse=True)[0][3]

BASELINE_EXP = OUTPUTS_ROOT / BASELINE_EXP_ID
if BASELINE_EXP_ID_ENV is None and not experiment_complete(BASELINE_EXP, require_full=(MAX_VIDEOS == 0)):
    discovered = discover_existing_baseline()
    if discovered is not None:
        BASELINE_EXP = discovered
        BASELINE_EXP_ID = discovered.name
        print("Auto-detected existing baseline:", BASELINE_EXP)

ENHANCED_EXP = OUTPUTS_ROOT / ENHANCED_EXP_ID

def run_checked(cmd: list[str], label: str) -> subprocess.CompletedProcess:
    print(f"Running {label}:", " ".join(str(x) for x in cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with exit code {result.returncode}. See output above.")
    return result

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SAM2_REPO_DIR:", SAM2_REPO_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("SAMPLE_SUBMISSION_ZIP:", SAMPLE_SUBMISSION_ZIP, "exists=", SAMPLE_SUBMISSION_ZIP.exists())
print("BASELINE_EXP:", BASELINE_EXP)
print("ENHANCED_EXP:", ENHANCED_EXP)
print("PUBLISH_ROOT:", PUBLISH_ROOT)
print("ARCHIVE_ROOT:", ARCHIVE_ROOT)
print("MAX_VIDEOS:", MAX_VIDEOS)
print("RESIZE_LONG_SIDE:", RESIZE_LONG_SIDE)
print("MAX_ANCHOR_RUNS:", MAX_ANCHOR_RUNS)
print("ANCHOR_FRACTIONS:", ANCHOR_FRACTIONS)
print("REQUIRE_ANCHOR_RERUN:", REQUIRE_ANCHOR_RERUN)

## 1. Runtime Check

In [ ]:
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found")

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"GPU memory total/free: {total_bytes / 1024**3:.2f}/{free_bytes / 1024**3:.2f} GiB")
except Exception as exc:
    print("torch check failed:", exc)

## 2. Install Dependencies

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "gdown", "opencv-python", "pillow", "numpy", "scipy", "scikit-image", "matplotlib", "tqdm", "pyyaml",
    "pycocotools", "imageio", "imageio-ffmpeg", "einops", "timm", "transformers", "accelerate",
    "hydra-core>=1.3.2", "iopath>=0.1.10",
], check=True)

def valid_sam2_repo(repo: Path) -> bool:
    required = [
        repo / "sam2" / "__init__.py",
        repo / "sam2" / "build_sam.py",
        repo / "sam2" / "sam2_video_predictor.py",
        repo / "pyproject.toml",
    ]
    return all(path.exists() and path.stat().st_size > 0 for path in required)

if SAM2_REPO_DIR.exists() and not valid_sam2_repo(SAM2_REPO_DIR):
    print("Removing incomplete local SAM2 repo:", SAM2_REPO_DIR)
    shutil.rmtree(SAM2_REPO_DIR)

if not valid_sam2_repo(SAM2_REPO_DIR):
    cached_repos = sorted(
        (
            repo
            for repo in OUTPUTS_ROOT.glob("*/external/sam2")
            if valid_sam2_repo(repo)
        ),
        key=lambda repo: repo.stat().st_mtime,
        reverse=True,
    )
    if cached_repos:
        SAM2_REPO_DIR = cached_repos[0].resolve()
        print("Reusing cached SAM2 repo directly:", SAM2_REPO_DIR)

if valid_sam2_repo(SAM2_REPO_DIR):
    print("SAM2 repo ready:", SAM2_REPO_DIR)
else:
    print("No valid cached SAM2 repo; enhanced runner will shallow-clone to:", SAM2_REPO_DIR)

sam2_import_check = [
    sys.executable,
    "-c",
    (
        "import sys; "
        f"sys.path.insert(0, {str(SAM2_REPO_DIR)!r}); "
        "import hydra, iopath, sam2; "
        "from sam2.build_sam import build_sam2_video_predictor; "
        "print('SAM2 builder import OK:', sam2.__file__)"
    ),
]
run_checked(sam2_import_check, "SAM2 dependency preflight")



## 3. Data Preparation

In [ ]:
if not IS_COLAB:
    raise RuntimeError("Dataset download is blocked outside Colab. Put the dataset under DATA_ROOT for local tests.")

import gdown

if DATA_ZIP.exists() and DATA_ZIP.stat().st_size > 0:
    print("Dataset zip exists:", DATA_ZIP)
else:
    gdown.download(url=DATA_URL, output=str(DATA_ZIP), fuzzy=True, quiet=False)
    if not DATA_ZIP.exists() or DATA_ZIP.stat().st_size == 0:
        raise RuntimeError(f"Dataset download failed: {DATA_ZIP}")

if any(DATA_ROOT.iterdir()):
    print("DATA_ROOT already populated:", DATA_ROOT)
else:
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(DATA_ROOT)
    print("Extracted to:", DATA_ROOT)

## 4. Run or Reuse SAM2.1 Baseline

In [ ]:
baseline_complete = experiment_complete(BASELINE_EXP, require_full=(MAX_VIDEOS == 0))
print("baseline_complete:", baseline_complete)

if not baseline_complete:
    cmd = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "run_baseline_sam2.py"),
        "--data-root", str(DATA_ROOT),
        "--output-dir", str(OUTPUTS_ROOT),
        "--experiment-id", BASELINE_EXP_ID,
        "--checkpoint", "sam2.1_hiera_large",
        "--sam2-repo-dir", str(SAM2_REPO_DIR),
        "--model-cfg", "configs/sam2.1/sam2.1_hiera_l.yaml",
        "--prompt-mode", "mask_box_points",
        "--resize-long-side", str(RESIZE_LONG_SIDE),
        "--skip-existing",
        "--save-overlays", "sample",
        "--make-submission",
    ]
    if SAMPLE_SUBMISSION_ZIP.exists():
        cmd.extend(["--sample-submission", str(SAMPLE_SUBMISSION_ZIP)])
    if MAX_VIDEOS:
        cmd.extend(["--max-videos", str(MAX_VIDEOS)])
    run_checked(cmd, "baseline")
    if not experiment_complete(BASELINE_EXP, require_full=(MAX_VIDEOS == 0)):
        raise RuntimeError(f"Baseline command finished, but expected artifacts are incomplete: {BASELINE_EXP}")
else:
    print("Reusing baseline:", BASELINE_EXP)

## 5. Enhanced Run: Pseudo-Anchor Fusion

In [ ]:
cmd = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "run_enhanced_sam2_anchors.py"),
    "--data-root", str(DATA_ROOT),
    "--baseline-exp", str(BASELINE_EXP),
    "--output-dir", str(OUTPUTS_ROOT),
    "--experiment-id", ENHANCED_EXP_ID,
    "--checkpoint", str(BASELINE_EXP / "checkpoints" / "sam2.1_hiera_large.pt") if (BASELINE_EXP / "checkpoints" / "sam2.1_hiera_large.pt").exists() else "sam2.1_hiera_large",
    "--sam2-repo-dir", str(SAM2_REPO_DIR),
    "--model-cfg", "configs/sam2.1/sam2.1_hiera_l.yaml",
    "--prompt-mode", "mask",
    "--resize-long-side", str(RESIZE_LONG_SIDE),
    "--anchor-fractions", ANCHOR_FRACTIONS,
    "--max-anchor-runs", str(MAX_ANCHOR_RUNS),
    "--anchor-search-radius", "8",
    "--anchor-quality-threshold", "0.25",
    "--fusion-tau", "48",
    "--baseline-floor", "0.35",
    "--anchor-scale", "0.95",
    "--foreground-threshold", "0.42",
    "--save-overlays", "sample",
    "--save-anchor-overlays", "none",
    "--make-submission",
]
if SAMPLE_SUBMISSION_ZIP.exists():
    cmd.extend(["--sample-submission", str(SAMPLE_SUBMISSION_ZIP)])
if MAX_VIDEOS:
    cmd.extend(["--max-videos", str(MAX_VIDEOS)])
if REQUIRE_ANCHOR_RERUN:
    cmd.append("--require-anchor-rerun")

run_checked(cmd, "enhanced")

publish_dir = PUBLISH_ROOT / ENHANCED_EXP_ID
publish_cmd = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "publish_run_for_codex.py"),
    "--exp-dir", str(ENHANCED_EXP),
    "--publish-dir", str(publish_dir),
    "--data-root", str(DATA_ROOT),
    "--baseline-exp", str(BASELINE_EXP),
    "--archive-dir", str(ARCHIVE_ROOT),
    "--replace",
]
if os.environ.get("SUFE_MAKE_FULL_ARCHIVE", "0").strip().lower() in {"1", "true", "yes"}:
    publish_cmd.append("--make-full-archive")
if os.environ.get("SUFE_MAKE_REVIEW_ZIP", "0").strip().lower() in {"1", "true", "yes"}:
    publish_cmd.append("--make-review-zip")
run_checked(publish_cmd, "publish-codex-review")
print("Codex review bundle:", publish_dir)


## 6. Inspect Outputs

In [ ]:
def show_json(path: Path, limit: int = 8000):
    print("\n==", path, "==")
    if not path.exists():
        print("missing")
        return
    text = path.read_text(encoding="utf-8")
    print(text[:limit])

show_json(ENHANCED_EXP / "sanity_check.json")
show_json(ENHANCED_EXP / "logs" / "per_video_status.json", limit=12000)
print("submission:", ENHANCED_EXP / "submission.zip")
print("diagnostics:", ENHANCED_EXP / "logs" / "self_diagnostics.csv")
print("fusion debug:", ENHANCED_EXP / "logs" / "fusion_debug.csv")

## Optional Ablation: Postprocess Only

Run this only if you want a lower-cost comparison. It does not launch pseudo-anchor SAM2 reruns.

In [ ]:
RUN_POSTPROCESS_ONLY = False

if RUN_POSTPROCESS_ONLY:
    pp_exp_id = f"sam2_postprocess_only_{dt.datetime.now():%Y%m%d_%H%M%S}"
    cmd = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "run_enhanced_sam2_anchors.py"),
        "--data-root", str(DATA_ROOT),
        "--baseline-exp", str(BASELINE_EXP),
        "--output-dir", str(OUTPUTS_ROOT),
        "--experiment-id", pp_exp_id,
        "--disable-anchor-rerun",
        "--save-overlays", "sample",
        "--save-anchor-overlays", "none",
        "--make-submission",
    ]
    if SAMPLE_SUBMISSION_ZIP.exists():
        cmd.extend(["--sample-submission", str(SAMPLE_SUBMISSION_ZIP)])
    if MAX_VIDEOS:
        cmd.extend(["--max-videos", str(MAX_VIDEOS)])
    run_checked(cmd, "postprocess-only")
    print("postprocess-only submission:", OUTPUTS_ROOT / pp_exp_id / "submission.zip")